# Air Quality Bronze — Auto Loader Pipeline

Ingests raw air-quality JSON snapshots from `/Volumes/weather_api/raw/raw_files/air_quality/` into `weather_api.bronze.air_quality_raw` using Auto Loader.

- Each JSON file is a single pretty-printed record (`multiLine=True`)
- Schema is inferred and evolves automatically with `rescue` mode
- `_rescued_data` column captures any fields outside the inferred schema
- `availableNow` trigger processes all pending files then stops — ideal for scheduled runs
- Ingestion metadata: `_ingest_timestamp`, `_source_file_path`

In [0]:
from pyspark.sql.functions import current_timestamp, col

# ── Configuration ──
SOURCE_PATH = "/Volumes/weather_api/raw/raw_files/air_quality"
SCHEMA_LOCATION = "/Volumes/weather_api/pipeline_state/autoloader_state/_schemas/air_quality"
CHECKPOINT_LOCATION = "/Volumes/weather_api/pipeline_state/autoloader_state/_checkpoints/air_quality"
TARGET_TABLE = "weather_api.bronze.air_quality_raw"

# ── Create schema if needed ──
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_api.bronze")

# ── Auto Loader: read raw JSON snapshots ──
# multiLine=True because files are pretty-printed (one JSON object per file)
# schemaEvolutionMode=rescue captures new/changed fields in _rescued_data
raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(SOURCE_PATH)
    .withColumn("_ingest_timestamp", current_timestamp())
    .withColumn("_source_file_path", col("_metadata.file_path"))
)

# ── Write to Delta table (incremental append) ──
# availableNow=True processes all pending files then stops the stream
# checkpointLocation ensures files are not reprocessed on restart
(raw_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
    .awaitTermination()
)

print(f"✅ Air Quality Bronze pipeline complete — data written to {TARGET_TABLE}")

In [0]:
# ── Validation ──
print(f"Row count: {spark.table(TARGET_TABLE).count()}")
print("\nSchema:")
spark.table(TARGET_TABLE).printSchema()

print("\nSample ingestion metadata:")
spark.table(TARGET_TABLE).select(
    "metadata.source",
    "metadata.location_name",
    "metadata.retrieval_timestamp_utc",
    "_ingest_timestamp",
    "_source_file_path",
).show(5, truncate=80)

print("\nRescued data (should be null if all fields matched schema):")
spark.table(TARGET_TABLE).select(
    "_ingest_timestamp",
    "_rescued_data",
).show(5, truncate=80)

print("\nCurrent air quality sample:")
spark.table(TARGET_TABLE).select(
    "metadata.location_name",
    "raw_response.current.pm2_5",
    "raw_response.current.pm10",
    "raw_response.current.us_aqi",
    "raw_response.current.ozone",
).show(5)